1. Представим, что у нас есть таблица "Employees" с
полями "Name", "Position", "Department", "Salary". \
● Создайте таблицу "Employees" с указанными полями.

In [39]:
from sqlite3 import connect, Connection, Cursor
from pprint import pprint

connection: Connection = connect("employees.db")
cursor: Cursor = connection.cursor()

cursor.execute("""CREATE TABLE IF NOT EXISTS Employees(
    Name varchar(30),
    Position varchar(30),
    Department varchar(30),
    Salary decimal(10,2)
)
""")

connection.commit()

Вставьте в таблицу несколько записей с информацией о
сотрудниках вашей компании.

In [40]:
employees = [
    ("Иван Иванов", "Менеджер", "Продажи", 65000.00),
    ("Анна Смирнова", "Разработчик", "ИТ", 95000.00),
    ("Петр Козлов", "Аналитик", "Маркетинг", 72000.00),
    ("Ольга Соколова", "HR-менеджер", "Кадры", 60000.00),
    ("Дмитрий Волков", "Техподдержка", "ИТ", 55000.00)
]

# Вставка данных
cursor.executemany("INSERT INTO Employees (Name, Position, Department, Salary) VALUES (?, ?, ?, ?)", employees)

connection.commit()

cursor.execute("SELECT * FROM Employees")
pprint(cursor.fetchall())

[('Иван Иванов', 'Менеджер', 'Продажи', 65000),
 ('Анна Смирнова', 'Разработчик', 'ИТ', 95000),
 ('Петр Козлов', 'Аналитик', 'Маркетинг', 72000),
 ('Ольга Соколова', 'HR-менеджер', 'Кадры', 60000),
 ('Дмитрий Волков', 'Техподдержка', 'ИТ', 55000)]


Измените данные в таблице для каких-то сотрудников.
Например, изменим должность одного из сотрудников на
более высокую.

In [41]:
cursor.execute("""
UPDATE Employees
SET Position = 'Руководитель отдела'
WHERE Name = 'Иван Иванов'
""")

connection.commit()

cursor.execute("SELECT * FROM Employees")
pprint(cursor.fetchall())

[('Иван Иванов', 'Руководитель отдела', 'Продажи', 65000),
 ('Анна Смирнова', 'Разработчик', 'ИТ', 95000),
 ('Петр Козлов', 'Аналитик', 'Маркетинг', 72000),
 ('Ольга Соколова', 'HR-менеджер', 'Кадры', 60000),
 ('Дмитрий Волков', 'Техподдержка', 'ИТ', 55000)]


Добавьте новое поле "HireDate" (дата приема на работу) в
таблицу "Employees".

In [42]:
cursor.execute("""
ALTER TABLE Employees
ADD COLUMN HireDate INT
""")

connection.commit()
cursor.execute("SELECT * FROM Employees")
pprint(cursor.fetchall())

[('Иван Иванов', 'Руководитель отдела', 'Продажи', 65000, None),
 ('Анна Смирнова', 'Разработчик', 'ИТ', 95000, None),
 ('Петр Козлов', 'Аналитик', 'Маркетинг', 72000, None),
 ('Ольга Соколова', 'HR-менеджер', 'Кадры', 60000, None),
 ('Дмитрий Волков', 'Техподдержка', 'ИТ', 55000, None)]


Добавьте записи о дате приема на работу для всех
сотрудников.

In [43]:
from datetime import datetime, timedelta
from random import randrange


def random_unix_timestamp():
    start_date = datetime(2019, 1, 1)
    end_date = datetime(2025, 12, 31)
    delta = end_date - start_date
    random_days = randrange(0, delta.days + 1)
    random_date = start_date + timedelta(days=random_days)
    return int(random_date.timestamp())

cursor.execute("SELECT Name FROM Employees")
employees = cursor.fetchall()

for (name,) in employees:
    timestamp = random_unix_timestamp()

    cursor.execute("""UPDATE Employees SET HireDate = ?
    WHERE Name = ?""", (timestamp, name))

connection.commit()

cursor.execute("SELECT * FROM Employees")
pprint(cursor.fetchall())

[('Иван Иванов', 'Руководитель отдела', 'Продажи', 65000, 1604955600),
 ('Анна Смирнова', 'Разработчик', 'ИТ', 95000, 1634245200),
 ('Петр Козлов', 'Аналитик', 'Маркетинг', 72000, 1650142800),
 ('Ольга Соколова', 'HR-менеджер', 'Кадры', 60000, 1734814800),
 ('Дмитрий Волков', 'Техподдержка', 'ИТ', 55000, 1637614800)]


Найдите всех сотрудников, чья должность "Manager".

In [44]:
# SQLite doesn't have funcs/procedures :/

cursor.execute("SELECT * FROM Employees WHERE Position = 'Менеджер'")
pprint(cursor.fetchall())

[]


Найдите всех сотрудников, у которых зарплата больше
5000 долларов.

In [45]:
cursor.execute("SELECT * FROM Employees WHERE Salary > (5000 * 12)")
pprint(cursor.fetchall())

[('Иван Иванов', 'Руководитель отдела', 'Продажи', 65000, 1604955600),
 ('Анна Смирнова', 'Разработчик', 'ИТ', 95000, 1634245200),
 ('Петр Козлов', 'Аналитик', 'Маркетинг', 72000, 1650142800)]


Найдите всех сотрудников, которые работают в отделе
"Sales".

In [46]:
cursor.execute("SELECT * FROM Employees WHERE Department = 'Продажи'")
pprint(cursor.fetchall())

[('Иван Иванов', 'Руководитель отдела', 'Продажи', 65000, 1604955600)]


Найдите среднюю зарплату по всем сотрудникам.

In [47]:
cursor.execute("SELECT AVG(Salary) FROM Employees")
pprint(cursor.fetchall())

[(69400.0,)]


Удалите таблицу "Employees".

In [48]:
cursor.execute("DROP TABLE Employees")
connection.commit()